Proyecto para deteccion de objetos por : 

-RECICLABLE---------------------------------------------------------------------------------------
Plásticos: Botellas de bebidas, envases de detergente, envases de alimentos (limpios).

Cartón: Cajas de cartón.

Vidrio: Botellas de bebidas y frascos.

Metales: Latas de aluminio (refrescos).

-NO RECICLABLE-------------------------------------------------------------------------------------
Unicel: Vasos, charolas, platos, empaques de electrodomésticos.

Plásticos de un solo uso: Bolsas, cubiertos desechables.

Residuos sanitarios: Pañales, cubrebocas.

In [ ]:
from tensorflow.keras.preprocessing.image import ImageDataGenerator
from tensorflow.keras.callbacks import EarlyStopping
from tensorflow.keras.applications import MobileNetV2
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Dense, Dropout, GlobalAveragePooling2D


In [10]:
# Rutas de imágenes
entrenamiento = 'CNN/Imagenes/TRAIN'
validacion = 'CNN/Imagenes/VALIDATION'

epocas          = 20
altura, anchura = 200, 200
batch_size      = 16
clases          = 2

# Generadores
entrenar = ImageDataGenerator(
    rescale=1/255,
    zoom_range=0.20,
    horizontal_flip=True,
    rotation_range=15,
    width_shift_range=0.1,
    height_shift_range=0.1,
    shear_range=0.1
)

validar = ImageDataGenerator(rescale=1/255)

imagenes_entrenamiento = entrenar.flow_from_directory(
    entrenamiento,
    target_size=(altura, anchura),
    batch_size=batch_size,
    class_mode='categorical',
    shuffle=True
)

imagenes_validacion = validar.flow_from_directory(
    validacion,
    target_size=(altura, anchura),
    batch_size=batch_size,
    class_mode='categorical',
    shuffle=False
)

#pasos
pasos            = imagenes_entrenamiento.samples // batch_size
pasos_validacion = imagenes_validacion.samples // batch_size

print("Clases:", imagenes_entrenamiento.class_indices)
print("Pasos entrenamiento:", pasos)
print("Pasos validación:", pasos_validacion)


Found 161 images belonging to 2 classes.
Found 35 images belonging to 2 classes.
Clases: {'N': 0, 'R': 1}
Pasos entrenamiento: 10
Pasos validación: 2


In [11]:
# MobileNetV2 ya fue entrenado con millones de imágenes
base = MobileNetV2(
    input_shape=(altura, anchura, 3),
    include_top=False,
    weights='imagenet'       # Pesos pre-entrenados
)
base.trainable = False       # No reentrenar la base

CNN = Sequential([
    base,
    GlobalAveragePooling2D(),
    Dense(256, activation='relu'),
    Dropout(0.5),
    Dense(clases, activation='softmax')
])

C:\Users\catyr\AppData\Local\Temp\ipykernel_17732\2789768743.py:2: UserWarning: `input_shape` is undefined or non-square, or `rows` is not in [96, 128, 160, 192, 224]. Weights for input shape (224, 224) will be loaded as the default.
  base = MobileNetV2(


In [13]:
CNN.compile(
    loss="categorical_crossentropy",
    optimizer="adam",
    metrics=["acc", "mse"]
)

historia = CNN.fit(
    imagenes_entrenamiento,
    validation_data=imagenes_validacion,
    epochs=epocas,
    steps_per_epoch=pasos,
    validation_steps=pasos_validacion,
    callbacks=[EarlyStopping(monitor='val_acc', patience=5, restore_best_weights=True)],
    verbose=1
)

# Ver si el modelo aprendió
#print("Precisión final entrenamiento:", historia.history['acc'][-1])
#print("Precisión final validación:", historia.history['val_acc'][-1])


print("Imágenes entrenamiento:", imagenes_entrenamiento.samples)
print("Imágenes validación:", imagenes_validacion.samples)
print("Pasos train:", pasos)
print("Pasos val:", pasos_validacion)

Epoch 1/20
10/10 ━━━━━━━━━━━━━━━━━━━━ 9s 451ms/step - acc: 0.9310 - loss: 0.1469 - mse: 0.0471 - val_acc: 0.9375 - val_loss: 0.2271 - val_mse: 0.0576
Epoch 2/20
10/10 ━━━━━━━━━━━━━━━━━━━━ 1s 39ms/step - acc: 0.9375 - loss: 0.0843 - mse: 0.0227 - val_acc: 0.9375 - val_loss: 0.3043 - val_mse: 0.0621
Epoch 3/20
10/10 ━━━━━━━━━━━━━━━━━━━━ 2s 226ms/step - acc: 0.9655 - loss: 0.0908 - mse: 0.0262 - val_acc: 0.9375 - val_loss: 0.3145 - val_mse: 0.0621
Epoch 4/20
10/10 ━━━━━━━━━━━━━━━━━━━━ 0s 38ms/step - acc: 1.0000 - loss: 0.0077 - mse: 1.8819e-04 - val_acc: 0.9375 - val_loss: 0.2869 - val_mse: 0.0616
Epoch 5/20
10/10 ━━━━━━━━━━━━━━━━━━━━ 2s 233ms/step - acc: 0.9724 - loss: 0.0752 - mse: 0.0220 - val_acc: 0.9375 - val_loss: 0.4064 - val_mse: 0.0640
Epoch 6/20
10/10 ━━━━━━━━━━━━━━━━━━━━ 1s 43ms/step - acc: 0.9375 - loss: 0.0597 - mse: 0.0197 - val_acc: 0.9375 - val_loss: 0.4007 - val_mse: 0.0634
Imágenes entrenamiento: 161
Imágenes validación: 35
Pasos train: 10
Pasos val: 2


In [15]:
import os
os.makedirs("CNN/Imagenes/Modelo", exist_ok=True)
CNN.save("CNN/Imagenes/Modelo/cnn.keras")
print("Modelo guardado")

Modelo guardado


In [16]:
import cv2
import numpy as np
from tensorflow.keras.utils import img_to_array
from keras.models import load_model
import os

ruta_modelo = "CNN/Imagenes/Modelo"
print("Archivos en la carpeta:", os.listdir(ruta_modelo))

# Cargar según el formato que exista
if os.path.exists(f"{ruta_modelo}/cnn.keras"):
    cnn = load_model(f"{ruta_modelo}/cnn.keras")
    print("Modelo .keras cargado")
elif os.path.exists(f"{ruta_modelo}/cnn.h5"):
    cnn = load_model(f"{ruta_modelo}/cnn.h5")
    print("Modelo .h5 cargado")
else:
    print("No se encontró ningún modelo, entrena primero")
    exit()

altura, anchura = 200, 200

capture = cv2.VideoCapture(0)

while True:
    ret, frame = capture.read()

    if not ret:
        print("Error al leer la cámara")
        break

    # Clasificar cada frame
    evaluar_imagen = cv2.cvtColor(frame, cv2.COLOR_BGR2RGB)
    evaluar_imagen = cv2.resize(evaluar_imagen, (anchura, altura))
    evaluar_imagen = img_to_array(evaluar_imagen)
    evaluar_imagen = evaluar_imagen / 255.0
    evaluar_imagen = np.expand_dims(evaluar_imagen, axis=0)

    clase = cnn.predict(evaluar_imagen, verbose=0)
    arg_max = np.argmax(clase[0])

    if arg_max == 0:
        resultado = "No reciclable"
        color = (0, 0, 255)    # Rojo
    else:
        resultado = "Reciclable"
        color = (0, 255, 0)    # Verde

    # Mostrar resultado en la cámara
    cv2.putText(frame, resultado,
                (20, 50),
                cv2.FONT_HERSHEY_SIMPLEX,
                1.5,
                color,
                3)

    # Mostrar probabilidades
    prob = f"N:{clase[0][0]:.2f}  R:{clase[0][1]:.2f}"
    cv2.putText(frame, prob,
                (20, 100),
                cv2.FONT_HERSHEY_SIMPLEX,
                0.8,
                (255, 255, 255),
                2)

    cv2.imshow("Camara", frame)

    k = cv2.waitKey(5)
    if k == 27:    # ESC para salir
        break

capture.release()
cv2.destroyAllWindows()

Archivos en la carpeta: ['cnn.keras', 'metricas.png']
Modelo .keras cargado
